# 00 - Colab setup and access check

Run this first in every new Colab session. It checks that Drive, Earth Engine and the MODIS collection all work (SRS OI-6, OI-10).

**Before running:** (1) push the repository to GitHub so the clone below finds the code; (2) in Colab's left sidebar, open *Secrets* (key icon) and add `EE_PROJECT` = your Earth Engine Cloud project id, with notebook access on. Never paste credentials into a cell.

If the repository is private, clone it with a token stored as a Colab secret instead of the URL below.

In [ ]:
import os, subprocess, sys
REPO_URL = 'https://github.com/Starpredetor/urban-heat-risk-prediction.git'
REPO_DIR = '/content/uhrm'
if os.path.isdir(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
sys.path.insert(0, REPO_DIR + '/src')
from uhrm.colab import bootstrap
project = bootstrap()

## Earth Engine and the MODIS collection
Confirms the collection id in `config/project.yaml` is valid, the band names exist, and shows the real date coverage (use it to settle OI-4, the last complete year).

In [ ]:
import ee
from datetime import datetime, timezone
print('Earth Engine OK:', ee.Number(1).add(1).getInfo() == 2)
col = ee.ImageCollection(project.modis.collection)
print('MOD11A2 composites available:', col.size().getInfo())
print('bands:', col.first().bandNames().getInfo())
t = col.aggregate_array('system:time_start')
first, last = t.reduce(ee.Reducer.min()).getInfo(), t.reduce(ee.Reducer.max()).getInfo()
as_date = lambda ms: datetime.fromtimestamp(ms / 1000, timezone.utc).date()
print('coverage:', as_date(first), 'to', as_date(last))

## Drive write test

In [ ]:
probe = project.path('_probe.txt')
probe.write_text('ok')
print('Drive write OK:', probe.read_text() == 'ok', '->', probe)
probe.unlink()